In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Загружаем необходимые библиотеки и расширения

# 0. импорты и библиотеки

In [ ]:
import pandas as pd
import numpy as np
import torch

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from nltk.tokenize import word_tokenize, TweetTokenizer
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split

# для решения задачи ML методами
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from catboost import CatBoostClassifier

from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, Trainer, TrainingArguments


In [ ]:
import random

def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
set_seed(42)

Загружаем данные

In [ ]:
train_df = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test_df = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")
sample_submission = pd.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")

train_df.head()


# 1. EDA, анализ, обработка пропусков

In [ ]:
# Общая инфомрация по таблице, пропуски
print("Shape:", train_df.shape)
print(train_df.info())
print(train_df.isnull().sum())
display(train_df.head(20))


In [ ]:
print(test_df.isnull().sum())

видим примерно одинаковое соотноешние пропусков в категориях для трейна и теста

In [ ]:

sns.countplot(x='target', data=train_df)
plt.title('Target Distribution')
plt.show()



In [ ]:
train_df['text_len'] = train_df['text'].apply(len)
train_df['word_count'] = train_df['text'].apply(lambda x: len(x.split()))
sns.histplot(train_df['text_len'], kde=True)
plt.title('Tweet Length Distribution')
plt.show()

test_df['text_len'] = test_df['text'].apply(len)
test_df['word_count'] = test_df['text'].apply(lambda x: len(x.split()))

In [ ]:
train_df[train_df['text_len'] > 140]


возможно более длинные твиты чаще содержат информацию о катастрофах, проверим эту гипотезу:

In [ ]:
train_df

In [ ]:
test_df

In [ ]:
train_df['text_len'] = train_df['text'].apply(len)
test_df['text_len'] = test_df['text'].apply(len)

# Пусть интервалы длины будут по 10 символов 
bins = list(range(0, 150, 10))
train_df['len_bin'] = pd.cut(train_df['text_len'], bins=bins)
test_df['len_bin'] = pd.cut(test_df['text_len'], bins=bins)

# Посчитаем долю твитов с target=1 в каждом интервале
length_analysis = train_df.groupby('len_bin')['target'].agg(['count', 'sum'])
length_analysis['disaster_ratio'] = length_analysis['sum'] / length_analysis['count']


# Визуализация
plt.figure(figsize=(10,5))
sns.barplot(x=length_analysis.index.astype(str), y=length_analysis['disaster_ratio'])
plt.xticks(rotation=45)
plt.ylabel('Доля твитов о катастрофах')
plt.title('Зависимость target=1 от длины твита')
plt.show()

In [ ]:
# средняя длина твита по классам
print('train_df')
print(train_df.groupby('target')['text_len'].mean())
print(train_df.groupby('target')['word_count'].mean())


видим что есть отличие по длине твита и количеству слов в трейне, попробуем добавить новые фичи:

In [ ]:
train_df['text_len'] = train_df['text'].apply(len)
train_df['word_count'] = train_df['text'].apply(lambda x: len(x.split()))

test_df['text_len'] = test_df['text'].apply(len)
test_df['word_count'] = test_df['text'].apply(lambda x: len(x.split()))


In [ ]:
train_df.info

In [ ]:
test_df.head(10)

In [ ]:
train_df.head(10)

In [ ]:
print("\nDisaster tweet:\n", train_df[train_df['target']==1]['text'].iloc[0])
print("\nNon-disaster tweet:\n", train_df[train_df['target']==0]['text'].iloc[0])

In [ ]:

import spacy
from tqdm.auto import tqdm

# Загружаем модель spaCy один раз
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Модель 'en_core_web_sm' не найдена. Установите: python -m spacy download en_core_web_sm")
    raise

def process_dataframe(input_df):
    """
    Применяет полный пайплайн обработки к датафрейму (тренировочному или тестовому).
    """
    print(f"Обработка датафрейма размером {input_df.shape}...")
    df = input_df.copy() # Работаем с копией, чтобы не изменять оригинал

    # 1. Создание бинарных признаков
    df['keyword_is_missing'] = df['keyword'].isnull().astype(int)
    df['location_is_missing'] = df['location'].isnull().astype(int)

    # 2. Восстановление 'location' с помощью NER
    locations_to_fill = df[df['location'].isnull()]
    if not locations_to_fill.empty:
        tqdm.pandas(desc="Поиск локаций")
        found_locations = locations_to_fill['text'].progress_apply(
            lambda text: next((ent.text for ent in nlp(text).ents if ent.label_ == 'GPE'), None)
        )
        df.loc[df['location'].isnull(), 'location'] = found_locations

    # 3. Восстановление 'keyword'
    # Используем уникальные ключевые слова только из ТРЕНИРОВОЧНОГО набора, чтобы избежать утечки данных
    unique_keywords = train_df['keyword'].dropna().unique().tolist()
    keywords_to_fill = df[df['keyword'].isnull()]
    if not keywords_to_fill.empty:
        def find_keyword_in_text(text):
            for keyword in unique_keywords:
                if re.search(r'\b' + re.escape(keyword) + r'\b', text, re.IGNORECASE):
                    return keyword
            return None
        tqdm.pandas(desc="Поиск ключевых слов")
        found_keywords = keywords_to_fill['text'].progress_apply(find_keyword_in_text)
        df.loc[df['keyword'].isnull(), 'keyword'] = found_keywords

    # 4. Финальная очистка
    df['keyword'].fillna('unknown', inplace=True)
    df['location'].fillna('unknown', inplace=True)

    # 5. Создание комбинированного поля
    df['combined_text'] = (
        df['text'] +
        ' [SEP] ' + df['keyword'] +
        ' [SEP] ' + df['location']
    )
    
    print("Обработка завершена.")
    return df

In [ ]:
# Загрузите ваши данные (если еще не загружены)
# train_df = pd.read_csv('train.csv')
# test_df = pd.read_csv('test.csv')

processed_train_df = process_dataframe(train_df)
processed_test_df = process_dataframe(test_df)

print("\n--- Сравнение результатов ---")
print("\nКолонки в обработанном ТРЕНИРОВОЧНОМ наборе:")
print(processed_train_df.columns)

print("\nКолонки в обработанном ТЕСТОВОМ наборе:")
print(processed_test_df.columns)

In [ ]:
processed_train_df.head()

Видим что много пропусков в стобцах location и keyword. Попробуем дозаполнить эти столбцы с помощью SpaCy

In [ ]:
# import spacy
# nlp = spacy.load('en_core_web_sm')

# def extract_location_from_tweet(text):
#     doc = nlp(text)
#     locations = []
#     for ent in doc.ents:
#         if ent.label_ == "GPE":
#             locations.append(ent.text)
#     return locations[0] if locations else None


# train_df['location_from_text'] = train_df['text'].apply(extract_location_from_tweet)
# #Затем можно использовать эту информацию для заполнения пропусков в 'location'

In [ ]:
# train_df.head(50)

видим что часть информации о локации удалось почерпнуть из текстов. Добавим информацию из текстов в колонку с location:

In [ ]:
# def combine_locations(row):
#     if pd.notna(row['location']) and row['location'].strip() != '':
#         return row['location']
#     elif pd.notna(row['location_from_text']) and row['location_from_text'].strip() != '':
#         return row['location_from_text']
#     else:
#         return np.nan  # или оставить пустым

# train_df['location'] = train_df.apply(combine_locations, axis=1)

In [ ]:
# train_df.head(50)

In [ ]:
# print(train_df.isnull().sum())

СТало лучше, но все равно многое не заполнено

In [ ]:
# # Выбираем только те строки, где столбец 'keyword' содержит NaN
# # pd.isna() аналогично pd.isnull() и часто используется для большей читаемости
# null_keywords_df = train_df[pd.isna(train_df['keyword'])]
# print("Строки с нулевыми значениями в 'keyword':")
# print(null_keywords_df.head()) # Выведем первые несколько строк
# print(f"Всего строк с нулевым 'keyword': {len(null_keywords_df)}")

In [ ]:
# # Выбираем только те строки, где столбец 'location' содержит NaN
# null_locations_df = train_df[pd.isna(train_df['location'])]
# print("\nСтроки с нулевыми значениями в 'location':")
# print(null_locations_df.head()) # Выведем первые несколько строк
# print(f"Всего строк с нулевым 'location': {len(null_locations_df)}")

In [ ]:
# # Используем оператор '&' (логическое И) для объединения условий
# both_null_df = train_df[pd.isna(train_df['keyword']) & pd.isna(train_df['location'])]
# print("\nСтроки, где 'keyword' и 'location' одновременно нулевые:")
# print(both_null_df.head())
# print(f"Всего строк, где оба столбца нулевые: {len(both_null_df)}")

Мета фичи

In [ ]:
import nltk
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words('english'))

In [ ]:
# word_count number of words in text
# unique_word_count number of unique words in text
# stop_word_count number of stop words in text
# url_count number of urls in text
# mean_word_length average character count in words
# char_count number of characters in text
# punctuation_count number of punctuations in text
# hashtag_count number of hashtags (#) in text
# mention_count number of mentions (@) in text

In [ ]:

def add_text_features(df, text_column='text'):
    
    df['word_count'] = df[text_column].apply(lambda x: len(str(x).split()))
    df['unique_word_count'] = df[text_column].apply(lambda x: len(set(str(x).split())))
    df['stop_word_count'] = df[text_column].apply(lambda x: len([w for w in str(x).lower().split() if w in STOPWORDS]))
    df['url_count'] = df[text_column].apply(lambda x: len([w for w in str(x).lower().split() if 'http' in w or 'https' in w]))
    df['mean_word_length'] = df[text_column].apply(lambda x: np.mean([len(w) for w in str(x).split()]) if len(str(x).split()) > 0 else 0)
    df['char_count'] = df[text_column].apply(lambda x: len(str(x)))
    df['punctuation_count'] = df[text_column].apply(lambda x: len([c for c in str(x) if c in string.punctuation]))
    df['hashtag_count'] = df[text_column].apply(lambda x: len([c for c in str(x) if c == '#']))
    df['mention_count'] = df[text_column].apply(lambda x: len([c for c in str(x) if c == '@']))
    return df

processed_train_df = add_text_features(processed_train_df, text_column='text')
processed_test_df = add_text_features(processed_test_df, text_column='text')

визуализируем распределение по подсчету мета фич:

In [ ]:
METAFEATURES = ['word_count', 'unique_word_count', 'stop_word_count', 'url_count', 'mean_word_length',
                'char_count', 'punctuation_count', 'hashtag_count', 'mention_count']
DISASTER_TWEETS = train_df['target'] == 1 


fig, axes = plt.subplots(ncols=2, nrows=len(METAFEATURES), figsize=(20, 50), dpi=100)

for i, feature in enumerate(METAFEATURES):

    sns.histplot(processed_train_df.loc[~DISASTER_TWEETS][feature], label='Not Disaster', ax=axes[i][0], color='green', kde=True)
    sns.histplot(processed_train_df.loc[DISASTER_TWEETS][feature], label='Disaster', ax=axes[i][0], color='red', kde=True)


    sns.histplot(processed_train_df[feature], label='Training', ax=axes[i][1], kde=True)
    sns.histplot(processed_train_df[feature], label='Test', ax=axes[i][1], kde=True)
    
    for j in range(2):
        axes[i][j].set_xlabel('')
        axes[i][j].tick_params(axis='x', labelsize=12)
        axes[i][j].tick_params(axis='y', labelsize=12)
        axes[i][j].legend()
        
    axes[i][0].set_title(f'{feature} Target Distribution in Training Set', fontsize=13)
    axes[i][1].set_title(f'{feature} Training & Test Set Distribution', fontsize=13)

plt.tight_layout() 
plt.show()

Видим что акие-то фичи могут быть не очень полезны для дальейшего прогноза, а какие-то впоследствии мы можем использовать (word_count, unique_word_count, stop_word_count, mean_word_length, char_count, punctuation_count)

На что можно дополнительно посмотреть перед токенизацией и лемматизацей:

In [ ]:
# ссылки, хештеги упоминания

# train_df['has_link'] = train_df['text'].str.contains('http').astype(int)
# train_df['has_mention'] = train_df['text'].str.contains('@').astype(int)
# train_df['has_hashtag'] = train_df['text'].str.contains('#').astype(int)

# print(train_df[['has_link', 'has_mention', 'has_hashtag']].mean())

# feature_means_by_target = train_df.groupby('target')[['has_link', 'has_mention', 'has_hashtag']].mean()
# print(feature_means_by_target)

In [ ]:

# feature_means_by_target.plot(kind='bar')
# plt.title('Среднее наличие ссылок, упоминаний и хэштегов по классам')
# plt.xlabel('target')
# plt.ylabel('Среднее значение (доля)')
# plt.xticks(rotation=0)
# plt.show()


In [ ]:
# частота эмодзи и спецсимволов

import emoji

def count_emojis(text):
    return sum(char in emoji.EMOJI_DATA for char in text)

train_df['emoji_count'] = train_df['text'].apply(count_emojis)
print(train_df['emoji_count'].mean())

emoji_by_target = train_df.groupby('target')[['emoji_count']].mean()
print(emoji_by_target)

In [ ]:
# частотный анализ слов до препроцессинга

from collections import Counter

# Для класса target=0
all_words_0 = ' '.join(processed_train_df[train_df['target'] == 0]['text']).lower().split()
word_freq_0 = Counter(all_words_0)

# Для класса target=1
all_words_1 = ' '.join(processed_train_df[train_df['target'] == 1]['text']).lower().split()
word_freq_1 = Counter(all_words_1)

print("Топ-20 слов для класса 0:")
print(word_freq_0.most_common(20))

print("\nТоп-20 слов для класса 1:")
print(word_freq_1.most_common(20))

In [ ]:
def plot_top_words(word_freq_input, title):
    
    if isinstance(word_freq_input, Counter):
        word_freq_list = word_freq_input.most_common(20)
    else:
        word_freq_list = word_freq_input 
    if not word_freq_list: 
        print("Нет данных для отображения.")
        return
    
    words, counts = zip(*word_freq_list)
    plt.figure(figsize=(10, 6))
    plt.barh(words[::-1], counts[::-1], color='skyblue')
    plt.title(title)
    plt.xlabel('Частота')
    plt.tight_layout()
    plt.show()

for cls in processed_train_df['target'].unique():
    all_words = ' '.join(processed_train_df[processed_train_df['target'] == cls]['text']).lower().split()
    word_freq = Counter(all_words)
    plot_top_words(word_freq, f'Топ-20 слов для класса {cls}')


In [ ]:
# облако слов

from wordcloud import WordCloud

disaster_text = ' '.join(processed_train_df[processed_train_df['target'] == 1]['text'])
non_disaster_text = ' '.join(processed_train_df[processed_train_df['target'] == 0]['text'])

plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.title("Disaster tweets")
plt.imshow(WordCloud(width=400, height=300).generate(disaster_text))
plt.axis('off')

plt.subplot(1,2,2)
plt.title("Non-disaster tweets")
plt.imshow(WordCloud(width=400, height=300).generate(non_disaster_text))
plt.axis('off')

plt.show()

In [ ]:
# повторы и дубликаты

duplicates = processed_train_df.duplicated('text').sum()
print(f"Повторяющихся твитов: {duplicates}")


In [ ]:
# Выделяем дубликаты с текстом и target
duplicates_df = processed_train_df[processed_train_df.duplicated('text', keep=False)]  # keep=False, чтобы взять все повторяющиеся строки

# Группируем по тексту, смотрим уникальные значения target
duplicate_targets = duplicates_df.groupby('text')['target'].nunique()

# Тексты, для которых меток больше одной (т.е. разный target для одного текста)
conflicting_targets = duplicate_targets[duplicate_targets > 1]

print(f"Количество дубликатов с разными метками: {len(conflicting_targets)}")

# Если хотите, посмотреть примеры таких конфликтов:
print(duplicates_df[duplicates_df['text'].isin(conflicting_targets.index)].sort_values('text'))


In [ ]:
duplicate_rows = processed_train_df[processed_train_df.duplicated('text', keep=False)].sort_values(by='text')

print("Дублирующиеся твиты с их метками:")

print(duplicate_rows[['text', 'target', 'id']].head(10)) # Show the first 10 for brevity

duplicates_with_mixed_targets = duplicate_rows.groupby('text')['target'].nunique()
mixed_target_texts = duplicates_with_mixed_targets[duplicates_with_mixed_targets > 1]

if not mixed_target_texts.empty:
    print("\nТвиты с одинаковым текстом, но разными метками (target):")

    for text_val in mixed_target_texts.index:
        print(f"\nТекст: {text_val}")
        print(processed_train_df[train_df['text'] == text_val][['text', 'target', 'id']])
else:
    print("\nОтсутствуют дублирующиеся твиты с одинаковым текстом, но разными метками.")

видим что из 110 повторяющихся твитов некоторые из дубликатов имеют противоречивые метки. Варианты обработки: или просто удалить 110 строк или заменить результат на медианный и отбросить дубликаты

In [ ]:
def get_majority_label(labels):
    return Counter(labels).most_common(1)[0][0]

# Создаем Series, где индекс - уникальный текст, значение - мажоритарная метка
majority_labels = processed_train_df.groupby('text')['target'].apply(get_majority_label)

# Обновляем DataFrame, заменяя все метки для повторяющихся текстов на мажоритарную
processed_train_df['target'] = processed_train_df['text'].map(majority_labels)

In [ ]:
# НЕ ЗАПУСКАТЬ!!!

# Получаем список текстов, которые имеют более одной уникальной метки
# texts_with_mixed_targets = train_df.groupby('text')['target'].nunique()
# texts_to_exclude = texts_with_mixed_targets[texts_with_mixed_targets > 1].index.tolist()

# Создаем новый DataFrame, исключая эти тексты
# train_df_cleaned = train_df[~train_df['text'].isin(texts_to_exclude)].copy()

# Теперь удаляем оставшиеся дубликаты (где текст совпадает, но метки тоже совпали)
# Это будут те 110 дубликатов минус те, которые были с противоречивыми метками
# train_df_cleaned.drop_duplicates(subset=['text'], keep='first', inplace=True)

# print(f"Оригинальный размер train_df: {len(train_df)}")
# print(f"Количество уникальных текстов с противоречивыми метками: {len(texts_to_exclude)}")
# print(f"Размер train_df после удаления конфликтующих и оставшихся дубликатов: {len(train_df_cleaned)}")

# Теперь вы можете продолжить работу с train_df_cleaned
# Если хотите, можете переприсвоить train_df = train_df_cleaned
# train_df = train_df_cleaned

In [ ]:
# НЕ ЗАПУСКАТЬ!!!

# Создаем копию DataFrame для работы
# train_df_processed = train_df.copy()

# Группируем по тексту и находим моду (наиболее частую метку)
# .mode()[0] берет первое значение моды, если их несколько
# resolved_targets = train_df_processed.groupby('text')['target'].agg(lambda x: x.mode()[0]).reset_index()
# resolved_targets.rename(columns={'target': 'resolved_target'}, inplace=True)

# Объединяем resolved_targets обратно с исходным DataFrame, чтобы обновить метки
# train_df_processed = train_df_processed.drop(columns=['target']).merge(resolved_targets, on='text', how='left')
# train_df_processed.rename(columns={'resolved_target': 'target'}, inplace=True)

# Теперь удаляем дубликаты по тексту, оставляя одну запись (с разрешенной меткой)
# train_df_processed.drop_duplicates(subset=['text'], keep='first', inplace=True)

# print(f"Оригинальный размер train_df: {len(train_df)}")
# print(f"Размер train_df после разрешения конфликтов и удаления дубликатов (с сохранением данных): {len(train_df_processed)}")

# Если вас устраивает результат, переприсвойте train_df
# train_df = train_df_processed

Далее посмотрим необычные слова в твитах, посчитаем их распредлеение по классам

In [ ]:

processed_train_df['text_lower'] = processed_train_df['text'].astype(str).str.lower()

rt_count = processed_train_df['text_lower'].apply(lambda x: 'rt' in x.split()).sum()
breaking_count = processed_train_df['text_lower'].apply(lambda x: 'breaking' in x.split()).sum()

total_tweets = len(processed_train_df)

print(f"Твиты, содержащие 'RT': {rt_count} ({rt_count / total_tweets:.2%})")
print(f"Твиты, содержащие 'breaking': {breaking_count} ({breaking_count / total_tweets:.2%})")


def get_word_counts(text_series, num_words=20):
    all_words = []
    
    for text in text_series:
        words = re.findall(r'\b\w+\b', text.lower()) # Find all word characters
        words = [word for word in words if word not in STOPWORDS and word not in string.punctuation and len(word) > 1]
        all_words.extend(words)
    return Counter(all_words).most_common(num_words)

print("\nСамые частые слова (после удаления стоп-слов и пунктуации):")
print(get_word_counts(processed_train_df['text']))

# Удалим временную колонку
processed_train_df.drop(columns=['text_lower'], inplace=True)

теперь будет снова интересно сравнить, как будет выглядеть облако слов и визуализировать наиболее распространенные слова

In [ ]:
for cls in processed_train_df['target'].unique():
    text_series = processed_train_df[processed_train_df['target'] == cls]['text']
    filtered_top_words = get_word_counts(text_series)
    plot_top_words(filtered_top_words, f'Топ-20 слов (без стоп-слов) для класса {cls}')



In [ ]:
from wordcloud import WordCloud

disaster_text = ' '.join(processed_train_df[processed_train_df['target'] == 1]['text'])
non_disaster_text = ' '.join(processed_train_df[processed_train_df['target'] == 0]['text'])

plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.title("Disaster tweets")
plt.imshow(WordCloud(width=400, height=300).generate(disaster_text))
plt.axis('off')

plt.subplot(1,2,2)
plt.title("Non-disaster tweets")
plt.imshow(WordCloud(width=400, height=300).generate(non_disaster_text))
plt.axis('off')

plt.show()

In [ ]:
processed_train_df['text_lower'] = processed_train_df['text'].astype(str).str.lower()

# Процент твитов с 'RT' в каждом классе
rt_counts_by_target = processed_train_df.groupby('target')['text_lower'].apply(lambda x: x.str.contains(r'\brt\b').sum())
rt_percentages_by_target = (rt_counts_by_target / processed_train_df.groupby('target').size()) * 100

print("\nПроцент твитов с 'RT' по классам:")
print(rt_percentages_by_target)

# Процент твитов с 'breaking' в каждом классе
breaking_counts_by_target = processed_train_df.groupby('target')['text_lower'].apply(lambda x: x.str.contains(r'\bbreaking\b').sum())
breaking_percentages_by_target = (breaking_counts_by_target / processed_train_df.groupby('target').size()) * 100

print("\nПроцент твитов с 'breaking' по классам:")
print(breaking_percentages_by_target)

# Удалим временную колонку
processed_train_df.drop(columns=['text_lower'], inplace=True)

In [ ]:
def is_mostly_special_chars(text, threshold_ratio=0.8):
    text_len = len(text)
    if text_len == 0:
        return False
    # Count non-alphanumeric characters (excluding spaces)
    special_char_count = len(re.findall(r'[^a-zA-Z0-9\s]', text))
    return special_char_count / text_len > threshold_ratio

processed_train_df['mostly_special_chars'] = processed_train_df['text'].apply(is_mostly_special_chars)
print(f"\nТвиты, состоящие почти полностью из спецсимволов: {processed_train_df['mostly_special_chars'].sum()}")

print("Примеры твитов, состоящих почти полностью из спецсимволов:")
print(processed_train_df[processed_train_df['mostly_special_chars']][['text', 'target']].head())
processed_train_df.drop(columns=['mostly_special_chars'], inplace=True) # Drop temporary column

def is_mostly_uppercase(text, threshold_ratio=0.8):
    text_len = len(text.replace(' ', '')) # Count only non-space characters
    if text_len == 0:
        return False
    uppercase_count = sum(1 for char in text if char.isupper())
    return uppercase_count / text_len > threshold_ratio

processed_train_df['mostly_uppercase'] = processed_train_df['text'].apply(is_mostly_uppercase)
print(f"\nТвиты, состоящие почти полностью из заглавных букв: {processed_train_df['mostly_uppercase'].sum()}")

print("Примеры твитов, состоящих почти полностью из заглавных букв:")
print(processed_train_df[processed_train_df['mostly_uppercase']][['text', 'target']].head(20))
processed_train_df.drop(columns=['mostly_uppercase'], inplace=True) # Удалим временную колонку

In [ ]:
import string

def calc_upper_ratio(text):
    if len(text) == 0:
        return 0
    upper_count = sum(1 for c in text if c.isupper())
    return upper_count / len(text)

def calc_special_ratio(text):
    if len(text) == 0:
        return 0
    special_count = sum(1 for c in text if c in string.punctuation)
    return special_count / len(text)

processed_train_df['upper_ratio'] = processed_train_df['text'].apply(calc_upper_ratio)
processed_train_df['special_ratio'] = processed_train_df['text'].apply(calc_special_ratio)

# Посмотрим примеры твитов с высокой долей заглавных или спецсимволов
print(processed_train_df[processed_train_df['upper_ratio'] > 0.7][['text', 'upper_ratio']].head(10))
print(processed_train_df[processed_train_df['special_ratio'] > 0.3][['text', 'special_ratio']].head(10))


In [ ]:

def count_special_chars_ratio(text):
    if not isinstance(text, str): # Проверка на нестроковые значения
        text = str(text)
    text_len = len(text)
    if text_len == 0:
        return 0.0
    # Считаем символы, которые не являются буквой, цифрой или пробелом
    special_char_count = len(re.findall(r'[^a-zA-Z0-9\s]', text))
    return special_char_count / text_len

def count_uppercase_ratio(text):
    if not isinstance(text, str): # Проверка на нестроковые значения
        text = str(text)
    text_no_space_len = len(text.replace(' ', ''))
    if text_no_space_len == 0:
        return 0.0
    uppercase_count = sum(1 for char in text if char.isupper())
    return uppercase_count / text_no_space_len

processed_train_df['special_char_ratio'] = processed_train_df['text'].apply(count_special_chars_ratio)
processed_test_df['special_char_ratio'] = processed_test_df['text'].apply(count_special_chars_ratio)

processed_train_df['uppercase_ratio'] = processed_train_df['text'].apply(count_uppercase_ratio)
processed_test_df['uppercase_ratio'] = processed_test_df['text'].apply(count_uppercase_ratio)

print("\nСтатистика по доле спецсимволов:")
print(processed_train_df['special_char_ratio'].describe())
print("\nСтатистика по доле заглавных букв:")
print(processed_train_df['uppercase_ratio'].describe())

In [ ]:
METAFEATURES.extend(['special_char_ratio', 'uppercase_ratio'])

METAFEATURES = ['word_count', 'unique_word_count', 'stop_word_count', 'url_count', 'mean_word_length',
                'char_count', 'punctuation_count', 'hashtag_count', 'mention_count']
DISASTER_TWEETS = processed_train_df['target'] == 1 


fig, axes = plt.subplots(ncols=2, nrows=len(METAFEATURES), figsize=(20, 50), dpi=100)

for i, feature in enumerate(METAFEATURES):

    sns.histplot(processed_train_df.loc[~DISASTER_TWEETS][feature], label='Not Disaster', ax=axes[i][0], color='green', kde=True)
    sns.histplot(processed_train_df.loc[DISASTER_TWEETS][feature], label='Disaster', ax=axes[i][0], color='red', kde=True)


    sns.histplot(processed_train_df[feature], label='Training', ax=axes[i][1], kde=True)
    sns.histplot(processed_test_df[feature], label='Test', ax=axes[i][1], kde=True)
    
    for j in range(2):
        axes[i][j].set_xlabel('')
        axes[i][j].tick_params(axis='x', labelsize=12)
        axes[i][j].tick_params(axis='y', labelsize=12)
        axes[i][j].legend()
        
    axes[i][0].set_title(f'{feature} Target Distribution in Training Set', fontsize=13)
    axes[i][1].set_title(f'{feature} Training & Test Set Distribution', fontsize=13)

plt.tight_layout() 
plt.show()

Посмотрим на количество заглавных букв в твитах

In [ ]:
# def get_uppercase_ratio(text):
#     total_chars = len(text)
#     if total_chars == 0:
#         return 0
#     return sum(1 for c in text if c.isupper()) / total_chars

# train_df['uppercase_ratio'] = train_df['text'].apply(get_uppercase_ratio)


In [ ]:
# # бинаризация text_len и word_count

# # Пороги потом поправим, сейчас сделаем как пример

# train_df['text_len_bin'] = (train_df['text_len'] > 100).astype(int)
# train_df['word_count_bin'] = (train_df['word_count'] > 15).astype(int)


In [ ]:
train_df.columns

In [ ]:
pd.set_option('display.max_columns', None) 

In [ ]:
processed_train_df.head(10)

In [ ]:
drop_cols = [
    'id',                   # неинформативен
    'text',                 # дублирует combined_text
    'char_count',           # дублирует text_len
    'upper_ratio',          # дублирует uppercase_ratio
    'special_ratio',        # дублирует special_char_ratio
    'url_count',            # дублирует has_link
    
]

processed_train_df.drop(columns=drop_cols, inplace=True)


In [ ]:
processed_train_df

# 2. Препроцессинг текста

базовая очистка

In [ ]:

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www.\S+", "", text)

    # Удаляем @упоминания и хэштеги (оставляем только слово)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)

    text = re.sub(r"[^\w\s]", "", text)

    # Токенизация
    tokens = word_tokenize(text)

    # Удаление стоп-слов и лемматизация
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    # Собираем обратно
    return " ".join(cleaned_tokens)

In [ ]:
processed_train_df['clean_text'] = processed_train_df['combined_text'].apply(preprocess_text)


лемматизация

In [ ]:
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    tokens = word_tokenize(text)
    lemmatized = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(lemmatized)

In [ ]:
processed_train_df['lemm_text'] = processed_train_df['clean_text'].apply(lemmatize_text)

# запускаем до сюда

In [ ]:
processed_train_df.head()

векторизация

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_text = tfidf.fit_transform(processed_train_df['lemm_text']) 


попробуем обучить модель без текстовых признаков, посмотрим на результаты

In [ ]:
num_features = ['text_len', 'word_count', 'keyword_is_missing', 'location_is_missing', 'unique_word_count', 'stop_word_count', 'mean_word_length', 'punctuation_count', 'hashtag_count', 'mention_count', 'special_char_ratio', 'uppercase_ratio']
X_num = processed_train_df[num_features]


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)


In [ ]:
from scipy.sparse import hstack

X_combined = hstack([X_text, X_num_scaled])
y = train_df['target']


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

model = LogisticRegression(max_iter=1000)
scores = cross_val_score(model, X_combined, y, cv=5, scoring='f1')
print(f"Средний F1-скор: {scores.mean():.10f}")


бейзлайн дал в целом предсказуемо невысокое предсказание без ключевых фичей с локацией и ключевыми словами

попробуем обучить catboost и добавим туда категориальные признаки

In [ ]:
# CatBoost

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, make_scorer

cat_features = ['keyword', 'location']
processed_train_df[cat_features] = processed_train_df[cat_features].fillna('nan').astype(str)

model = CatBoostClassifier(verbose=100)
train_pool = Pool(processed_train_df[cat_features + num_features], label=train_df['target'], cat_features=cat_features)
model.fit(train_pool)

scores = cross_val_score(model, X_combined, y, cv=5, scoring='f1')
print(f"Средний F1-скор: {scores.mean():.10f}")

Средний F1-скор: 0.5280521463

In [ ]:
from sklearn.metrics import f1_score, make_scorer

cat_model = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='F1',
    random_seed=42,
    verbose=0, # Отключаем подробный вывод в процессе GridSearch, чтобы не засорять
    early_stopping_rounds=50 # Может помочь ускорить, но для каждой комбинации в GridSearch
                             # будет своя остановка, что может быть медленно.
                             # Возможно, лучше убрать для GridSearch, если не хватает ресурсов.
)

param_grid = {
    # 'iterations': [100, 200, 300], # Количество деревьев
    # 'learning_rate': [0.03, 0.05, 0.1], # Скорость обучения
    # 'depth': [4, 6, 8], # Максимальная глубина деревьев
    'iterations': [300], # Количество деревьев
    'learning_rate': [0.1, 0.2], # Скорость обучения
    'depth': [8], # Максимальная глубина деревьев
    'l2_leaf_reg': [1, 3, 5], # L2 регуляризация
    'border_count': [32, 128], # Количество порогов для числовых признаков
}

scorer = make_scorer(f1_score)

# Инициализируем GridSearchCV

grid_search = GridSearchCV(
    estimator=cat_model,
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=42), 
    scoring=scorer,
    n_jobs=-1, # Используем все доступные ядра
    verbose=2 # Выводим информацию о ходе выполнения
)

print("\nЗапуск GridSearchCV...")
grid_search.fit(X_combined, y)

print("\nGridSearchCV завершен.")

print(f"\nЛучшие параметры: {grid_search.best_params_}")
print(f"Лучший F1-скор на валидации (из GridSearchCV): {grid_search.best_score_:.10f}")

best_catboost_model = grid_search.best_estimator_

лучшие результаты:

Лучшие параметры: {'depth': 8, 'iterations': 300, 'learning_rate': 0.1}

Лучший F1-скор на валидации (из GridSearchCV): 0.7248071728

In [ ]:
from sklearn.metrics import f1_score, make_scorer

In [ ]:
from sklearn.metrics import f1_score, make_scorer

cat_model = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='F1',
    random_seed=42,
    verbose=0, # Отключаем подробный вывод в процессе GridSearch, чтобы не засорять
    early_stopping_rounds=50 # Может помочь ускорить, но для каждой комбинации в GridSearch
                             # будет своя остановка, что может быть медленно.
                             # Возможно, лучше убрать для GridSearch, если не хватает ресурсов.
)

param_grid = {
    # 'iterations': [100, 200, 300], # Количество деревьев
    # 'learning_rate': [0.03, 0.05, 0.1], # Скорость обучения
    # 'depth': [4, 6, 8], # Максимальная глубина деревьев
    'iterations': [300], # Количество деревьев
    'learning_rate': [0.1, 0.2], # Скорость обучения
    'depth': [8], # Максимальная глубина деревьев
    'l2_leaf_reg': [1, 3, 5], # L2 регуляризация
    'border_count': [32, 128], # Количество порогов для числовых признаков
}

scorer = make_scorer(f1_score)

# Инициализируем GridSearchCV

grid_search = GridSearchCV(
    estimator=cat_model,
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=42), 
    scoring=scorer,
    n_jobs=-1, # Используем все доступные ядра
    verbose=2 # Выводим информацию о ходе выполнения
)

print("\nЗапуск GridSearchCV...")
grid_search.fit(X_combined, y)

print("\nGridSearchCV завершен.")

print(f"\nЛучшие параметры: {grid_search.best_params_}")
print(f"Лучший F1-скор на валидации (из GridSearchCV): {grid_search.best_score_:.10f}")

best_catboost_model = grid_search.best_estimator_

Запуск GridSearchCV...

Fitting 4 folds for each of 12 candidates, totalling 48 fits


GridSearchCV завершен.


Лучшие параметры: {'border_count': 128, 'depth': 8, 'iterations': 300, 'l2_leaf_reg': 3, 'learning_rate': 0.2}

Лучший F1-скор на валидации (из GridSearchCV): 0.7295943838

In [ ]:
processed_train_df

# 3. Использование BERT

In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"  # или другую
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Токенизация
encoded = tokenizer(
    list(processed_train_df['lemm_text']), 
    padding=True, 
    truncation=True, 
    return_tensors="pt"
)

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained(model_name)
model.eval()
with torch.no_grad(): # with torch.inference_mode():
    # model.train() будет ли ошибка?
    outputs = model(**encoded)
    cls_embeddings = outputs.last_hidden_state[:, 0, :]  # [CLS] токен


попробуем использовать полученные эмбеддинги для ML моделей

In [ ]:
# X_train, X_val, y_train, y_val = train_test_split(X_bert, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, make_scorer

X_bert = cls_embeddings.numpy()  
y = processed_train_df['target'].values



In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_bert, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
print("F1-score:", f1_score(y_val, y_pred))

F1-score: 0.7856560717196414

с полученными эмбеддингами получили скор гораздо выше, при том что использовалась бейзлайн модель из коробки без тьюнинга гиперпараметров. Попробуем использовать эмбеддинги в catboost classifier

In [ ]:
cat_model = CatBoostClassifier(
    iterations=300,
    depth=8,
    learning_rate=0.1,
    eval_metric='F1',
    random_seed=42,
    verbose=0,
    border_count = 128,
    l2_leaf_reg = 3
)


cat_model.fit(X_train, y_train)
y_pred = cat_model.predict(X_val)
print("F1-score:", f1_score(y_val, y_pred))

F1-score: 0.7773212818405917

Добавим категориальные и числовые признаки, чтобы посмотреть, как это повлияет на предсказание модели

In [ ]:


X_features = processed_train_df[num_features].values
X_combined = np.hstack((X_bert, X_features))


X_train, X_val, y_train, y_val = train_test_split(X_combined, y, test_size=0.2, random_state=42, stratify=y)


cat_model.fit(X_train, y_train)
y_pred = cat_model.predict(X_val)
print("F1-score:", f1_score(y_val, y_pred))


F1-score: 0.7813798836242726

Здесь попробуем использовать категроиальные и числовые признаки

In [ ]:
X_cat = processed_train_df[['keyword', 'location']].astype(str)
y = processed_train_df['target'].values

bert_cols = [f'bert_{i}' for i in range(X_bert.shape[1])]
df_bert = pd.DataFrame(X_bert, columns=bert_cols)

X_all = pd.concat([df_bert, pd.DataFrame(X_num, columns=num_features), X_cat.reset_index(drop=True)], axis=1)

X_train, X_val, y_train, y_val = train_test_split(
    X_all, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
cat_features = ['keyword', 'location']

cat_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=8,
    cat_features=cat_features,  # указали категориальные
    eval_metric='F1',
    random_seed=42,
    verbose=0,
    border_count = 128,
    l2_leaf_reg = 3
)

cat_model.fit(X_train, y_train)
y_pred = cat_model.predict(X_val)
print("F1-score:", f1_score(y_val, y_pred))

попробуем другие BERT модели и сравним их результаты

In [ ]:

# Выбираем roberta
MODEL_NAME = "cardiffnlp/twitter-roberta-base"

# Загружаем токенизатор и модель
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

texts = processed_train_df['lemm_text'].tolist()

In [ ]:
def get_roberta_embeddings(texts, tokenizer, model, max_len=128, batch_size=32):
    model.eval()
    embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i:i+batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
            output = model(**encoded)
            last_hidden_state = output.last_hidden_state  # (batch, seq_len, hidden)
            mean_embeddings = last_hidden_state.mean(dim=1)  # (batch, hidden)
            embeddings.append(mean_embeddings)
    
    return torch.cat(embeddings).numpy()


In [ ]:
X_roberta = get_roberta_embeddings(texts, tokenizer, model)


In [ ]:
# def train_and_evaluate_model(model, train_df, target_column, num_features, cat_features=None, bert_features=None, test_size=0.2, random_state=42):
#     """
#     Унифицированная функция для обучения и оценки модели CatBoost.

#     Аргументы:
#     model: Экземпляр CatBoostClassifier.
#     train_df (pd.DataFrame): DataFrame с обучающими данными.
#     target_column (str): Название столбца с целевой переменной (метками).
#     num_features (list): Список названий столбцов с числовыми признаками.
#     cat_features (list, optional): Список названий столбцов с категориальными признаками.
#                                   По умолчанию None.
#     bert_features (np.ndarray, optional): Массив признаков BERT (например, X_bert).
#                                         Предполагается, что он уже выровнен по строкам с train_df.
#                                         По умолчанию None.
#     test_size (float, optional): Доля данных для валидационного набора. По умолчанию 0.2.
#     random_state (int, optional): Зерно для генератора случайных чисел для воспроизводимости. По умолчанию 42.

#     Возвращает:
#     tuple: (обученная_модель, f1_score_на_валидации, y_pred)
#     """

#     # Подготовка целевой переменной
#     y = train_df[target_column].values

#     # Извлечение числовых признаков
#     X_features = train_df[num_features].values

#     # Объединение признаков
#     if bert_features is not None:
        
#         if bert_features.shape[0] != train_df.shape[0]:
#             raise ValueError("Количество строк в bert_features не совпадает с train_df.")
#         X_combined = np.hstack((bert_features, X_features))
#     else:
#         X_combined = X_features

#     # Определение категориальных признаков для CatBoost (если есть)
#     # Важно: индексы категориальных признаков должны быть относительно ОБЪЕДИНЕННОГО набора X_combined
#     cat_features_indices = []
   

#     X_train_df = train_df[num_features].copy() # Создаем копию, чтобы избежать SettingWithCopyWarning
    
#     if cat_features is not None:
#         # Убедимся, что категориальные признаки корректно обработаны (NaN и тип string)
#         # Это должен быть **важный шаг, выполненный ДО вызова этой функции**,
#         # как вы уже указали в своей строке:
#         # train_df[cat_features] = train_df[cat_features].fillna('nan').astype(str)
#         for col in cat_features:
#             if col in train_df.columns:
#                 X_train_df[col] = train_df[col]
#             else:
#                 raise ValueError(f"Категориальный признак '{col}' не найден в train_df.")

#     if bert_features is not None:
#         # Добавляем BERT признаки как новые столбцы.
#         # Это может быть неэффективно для очень большого количества BERT признаков.
#         # В таком случае, лучше передавать индексы столбцов для CatBoost.
#         # Но для большинства случаев это работает.
#         bert_df = pd.DataFrame(bert_features, index=train_df.index)
#         bert_column_names = [f'bert_feature_{i}' for i in range(bert_features.shape[1])]
#         bert_df.columns = bert_column_names
#         X_train_df = pd.concat([X_train_df, bert_df], axis=1)

#     # Определяем категориальные признаки для CatBoost по их названиям
#     # CatBoost сам определит, какие столбцы являются категориальными, если их dtype - 'category' или 'object'.
#     # Но явно указать их через cat_features (список имен столбцов) лучше.
#     final_cat_features = cat_features if cat_features is not None else []
    
#     # Для CatBoost, если вы передаете DataFrame, он может автоматически определить категориальные столбцы
#     # если они имеют dtype 'object' или 'category'.
#     # Однако, явное указание `cat_features` в `fit` всегда надежнее.

#     # Разделение данных на обучающую и валидационную выборки
#     # Передаем весь DataFrame X_train_df, а не только numpy array.
#     X_train, X_val, y_train, y_val = train_test_split(X_train_df, y, test_size=test_size, random_state=random_state, stratify=y)

#     # Обучение модели
#     # Если cat_features были переданы, CatBoost их использует.
#     # Если train_df[cat_features] уже `astype(str)` или `astype('category')`, CatBoost их распознает.
#     model.fit(X_train, y_train, cat_features=final_cat_features, verbose=False) # verbose=False здесь, так как функция сама контролирует вывод

#     # Предсказание и оценка
#     y_pred = model.predict(X_val)
#     f1 = f1_score(y_val, y_pred)

#     print(f"Модель: {model.__class__.__name__}")
#     print(f"Признаки (числовые): {num_features}")
#     if cat_features:
#         print(f"Признаки (категориальные): {cat_features}")
#     if bert_features is not None:
#         print(f"Использованы BERT признаки: Да (размер: {bert_features.shape[1]})")
#     print(f"F1-score на валидации: {f1:.4f}")

#     return model, f1, y_pred

In [ ]:
y = processed_train_df['target'].values

X_train, X_val, y_train, y_val = train_test_split(X_roberta, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


y_pred = model.predict(X_val)
print("F1-score:", f1_score(y_val, y_pred))


RoBERTa и разные вариации CatBoost (только эмбеддинги, эмбеддинги + числовые, эмбеддинги числовые и категориальные)

CatBoost + RoBERTa эмбеддинги

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_roberta, y, test_size=0.2, random_state=42, stratify=y)

cat_model = CatBoostClassifier(
    iterations=300,
    depth=8,
    learning_rate=0.1,
    eval_metric='F1',
    random_seed=42,
    verbose=0,
    border_count = 128,
    l2_leaf_reg = 3
)


cat_model.fit(X_train, y_train)
y_pred = cat_model.predict(X_val)
print("F1-score:", f1_score(y_val, y_pred))


CatBoost, RoBERTA + числовые признаки

In [ ]:
X_features = processed_train_df[num_features].values
X_combined = np.hstack((X_roberta, X_features))


X_train, X_val, y_train, y_val = train_test_split(X_combined, y, test_size=0.2, random_state=42, stratify=y)

cat_model = CatBoostClassifier(
    iterations=300,
    depth=8,
    learning_rate=0.1,
    eval_metric='F1',
    random_seed=42,
    verbose=0,
    border_count = 128,
    l2_leaf_reg = 3
)


cat_model.fit(X_train, y_train)
y_pred = cat_model.predict(X_val)
print("F1-score:", f1_score(y_val, y_pred))

CatBoost + RoBERTa эмбеддинги, числовые и категориальные признаки

In [ ]:
X_cat = processed_train_df[['keyword', 'location']].astype(str)
y = processed_train_df['target'].values

bert_cols = [f'bert_{i}' for i in range(X_roberta.shape[1])]
df_roberta = pd.DataFrame(X_roberta, columns=bert_cols)

X_all = pd.concat([df_bert, pd.DataFrame(X_num, columns=num_features), X_cat.reset_index(drop=True)], axis=1)

X_train, X_val, y_train, y_val = train_test_split(
    X_all, y, test_size=0.2, random_state=42, stratify=y
)

cat_features = ['keyword', 'location']

cat_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=8,
    cat_features=cat_features,  # указали категориальные
    eval_metric='F1',
    random_seed=42,
    verbose=0,
    border_count = 128,
    l2_leaf_reg = 3
)

cat_model.fit(X_train, y_train)
y_pred = cat_model.predict(X_val)
print("F1-score:", f1_score(y_val, y_pred))

# 4. Finetuning BERT

Finetunung будет состоять из следующих этапов:
1. выберем модель для дообучения (cardiffnlp/twitter-roberta-base - тк она адаптирована под твиты)
2. подготовим данные (класс для обработки датасета, разбиение данных на val/test, токенизация)
3. Используем AutoModelForSequenceClassification
4. Обучим модель 
5. оценим результаты (f1-score, confusion matrix, сравнение с предыдущими результатами)
6. предскажем результаты на тестовой выборке и можно сабмитить


In [ ]:
# !pip install -U transformers
# !pip install -U datasets

In [ ]:
# !pip install transformers datasets evaluate

In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
from transformers import TrainingArguments

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# наша модель для дообучения

MODEL_NAME = 'cardiffnlp/twitter-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)


In [ ]:
processed_train_df

In [ ]:
# собираем датсет только из нужных данных

# df = processed_train_df[['lemm_text', 'target']].copy()
# df['text'] = df['lemm_text'].astype(str)
# df['label'] = df['target']
# df = df[['text', 'label']]

# X = processed_train_df['lemm_text'].values
# y = processed_train_df['target'].values

# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


train_texts, val_texts, train_labels, val_labels = train_test_split(
    processed_train_df['clean_text'].tolist(), # поменяли и попробуем использовать текст без предобработки
    processed_train_df['target'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=processed_train_df['target'] 
)

In [ ]:
train_dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels})
val_dataset = Dataset.from_dict({'text': val_texts, 'label': val_labels})

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_val_dataset = val_dataset.map(tokenize_function, batched=True)

tokenized_train_dataset = tokenized_train_dataset.remove_columns(["text"]).rename_column("label", "labels")
tokenized_val_dataset = tokenized_val_dataset.remove_columns(["text"]).rename_column("label", "labels")

tokenized_train_dataset.set_format("torch")
tokenized_val_dataset.set_format("torch")

print("Пример обработанной записи:")
print(tokenized_train_dataset[0])


дебажим

In [ ]:
# DEBUG = False

# if DEBUG:
#     tokenized_train_dataset = tokenized_train_dataset.select(range(200))  # первые 200 примеров
#     tokenized_val_dataset = tokenized_val_dataset.select(range(100))      # первые 100 примеров

#     print(f"⚠️ Тренируемся только на части данных: {len(tokenized_train_dataset)} train / {len(tokenized_val_dataset)} val")


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    acc = accuracy_score(labels, predictions)
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
print(f"Размер train: {len(tokenized_train_dataset)}")
print(f"Размер val: {len(tokenized_val_dataset)}")

import torch
print(f"CUDA доступна: {torch.cuda.is_available()}")
print(f"Устройство модели: {next(model.parameters()).device}")



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

Дебаг на батче

In [ ]:
# from transformers import Trainer, TrainingArguments
# from transformers import logging

# logging.set_verbosity_info()

# # параметры для дебага
# training_args = TrainingArguments(
#     output_dir='./results',
#     num_train_epochs=3,  # пока 1 эпоха
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     warmup_steps=50,
#     weight_decay=0.01,
#     logging_dir='./logs',
#     logging_steps=1, 
#     eval_strategy="steps",
#     eval_steps=15,     # нечастая валидация
#     save_strategy="no",  # не сохраняем, пока тестим
#     report_to="none",  # отключаем WandB
#     disable_tqdm=False
# )



# # параметры для настоящего обучения
# # training_args = TrainingArguments(
# #     output_dir='./results',
# #     num_train_epochs=2,
# #     per_device_train_batch_size=16,
# #     per_device_eval_batch_size=16,
# #     warmup_steps=500,
# #     weight_decay=0.01,
# #     logging_dir='./logs',
# #     logging_steps=100,
# #     eval_strategy="steps",
# #     eval_steps=200,
# #     save_strategy="steps",
# #     save_steps=200,
# #     load_best_model_at_end=True,
# #     metric_for_best_model="f1",
# #     greater_is_better=True,
# #     report_to="none",  # отключаем wandb/tensorboard
# # )


# data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# # параметры для настоящего обучения
# # trainer = Trainer(
# #     model=model,
# #     args=training_args,
# #     train_dataset=tokenized_train_dataset,
# #     eval_dataset=tokenized_val_dataset,
# #     tokenizer=tokenizer,
# #     data_collator=data_collator,
# #     compute_metrics=compute_metrics
# # )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_train_dataset,
#     eval_dataset=tokenized_val_dataset,
#     tokenizer=tokenizer,
#     data_collator=data_collator,
#     compute_metrics=compute_metrics,
    
# )

# trainer.train()

# # # Запускаем дообучение!
# # trainer.train()

Полноценное обучение

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,  
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_strategy="steps",
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="f1",  
    greater_is_better=False,
    report_to="none",
    disable_tqdm=False,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics, # метрики
)

trainer.train()


In [ ]:

print("Проводим оценку на валидационном наборе...")
eval_results = trainer.evaluate()
print(f"Результаты оценки: {eval_results}")

print("Подготовка тестовых данных для предсказания...")
test_dataset = Dataset.from_dict({'text': test_df['text'].tolist()})

tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = tokenized_test_dataset.remove_columns(["text"])
tokenized_test_dataset.set_format("torch")

print("Получение предсказаний...")
predictions = trainer.predict(tokenized_test_dataset)
predicted_labels = np.argmax(predictions.predictions, axis=1)


print("Создание файла для сабмита...")
submission_df = pd.DataFrame({'id': test_df['id'], 'target': predicted_labels})
submission_df.to_csv('submission.csv', index=False)

print("\nФайл submission_bert.csv успешно создан!")
display(submission_df.head())

